# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the [FAIR^2 Mass Spectrometry](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library and the Croissant metadata standard.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)

# Access metadata (do not subscript or iterate as dictionary)
print("Dataset Title:", dataset.metadata.name)
print("Description:", dataset.metadata.description)
print("---")
print("Published:", getattr(dataset.metadata, 'datePublished', 'N/A'))

## 2. Data Overview
Review available record sets, fields, and their `@id`s. This is important, as Croissant datasets use these `@id` fields for referencing all key entities.

In [ ]:
# List all record sets with their @id and field info
record_sets = list(dataset.metadata.record_sets)
print(f"Record sets found in dataset (by @id):\n")
for rs in record_sets:
    print(f"- Record set @id: {rs['@id']}")
    print(f"  Name: {rs.get('name', '')}")
    if 'field' in rs and rs['field']:
        fields = rs['field']
        print(f"  Fields:")
        for f in fields:
            if isinstance(f, dict):
                print(f"    - {f.get('@id','')} ({f.get('name','')}, type: {f.get('dataType','')})")
            else:
                print(f"    - {f}") # may just be @id string
    print()

## 3. Data Extraction
Load data from the record set(s) of interest into pandas DataFrames for analysis. All references are made using the entity `@id`s.

For this dataset, the main record set typically contains the protein measurement table. See above cell for record set `@id` (example: `_:main_table` or similar).

In [ ]:
# Replace this with the main record set @id from the overview cell above
# Let's fetch all record set @ids first
record_set_ids = [rs["@id"] for rs in dataset.metadata.record_sets]
print(f"Found record set @ids: {record_set_ids}")

# Select main record set (by convention, the table is the first record set)
main_record_set_id = record_set_ids[0]
print(f"Working with record set: {main_record_set_id}")

# Load records from the main record set
records = list(dataset.records(record_set=main_record_set_id))
df = pd.DataFrame(records)
print(f"Loaded {len(df)} records. Columns available:")
print(df.columns.tolist())
df.head()

## 4. Exploratory Data Analysis (EDA)
Now, let's perform some EDA tasks:
- View distribution and statistics of numeric columns
- Filter and normalize one numeric field
- Group by a categorical variable if present

Fields are always referenced via their `@id`. You can look up these field IDs from the earlier overview.

In [ ]:
import numpy as np

# Choose a numeric field for demonstration (replace with a valid numeric field @id from section 2 overview)
# Examples: 'cr:peptide_count', 'cr:molecular_weight', etc. Use the IDs listed previously.
numeric_fields = [col for col in df.columns if df[col].dtype in [np.float64, np.int64] or pd.api.types.is_numeric_dtype(df[col])]
print(f"Numeric columns detected: {numeric_fields}")

if numeric_fields:
    numeric_field = numeric_fields[0]
    print(f"Analyzing numeric field by @id: {numeric_field}")
    
    threshold = df[numeric_field].mean()  # Use mean as filter threshold example
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold:.2f} (n={len(filtered_df)}):")
    print(filtered_df.head())

    # Normalization (z-score)
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} (z-score):")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try grouping by a categorical field if available (e.g., protein type, accession)
    categorical_fields = [col for col in df.columns if df[col].dtype == object and col != numeric_field]
    if categorical_fields:
        group_field = categorical_fields[0]
        print(f"Grouping by {group_field} (by @id)...")
        grouped = filtered_df.groupby(group_field)[numeric_field].mean()
        print(grouped.head())
else:
    print("No numeric fields detected for EDA.")

## 5. Visualization
Visualize distribution of the selected numeric value, and relationship with a categorical field if available.
You may need matplotlib or seaborn. Let's try basic histogram and boxplot.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if 'numeric_field' in locals() and numeric_field in df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), bins=30)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()
    
    # Boxplot by group_field if available
    if 'group_field' in locals() and group_field in df.columns and df[group_field].nunique() < 20:
        plt.figure(figsize=(10,6))
        sns.boxplot(data=df, x=group_field, y=numeric_field)
        plt.xticks(rotation=45)
        plt.title(f"{numeric_field} by {group_field}")
        plt.show()
else:
    print("No suitable numeric field for visualization.")

## 6. Conclusion

- Using the Croissant schema and `mlcroissant`, we explored a complex scientific data package and demonstrated access to tabular data using machine-actionable metadata.
- Most operations required referencing entities by their `@id`—this is central to the FAIR principle and enables robust, standardized data access.
- We performed basic EDA, including normalization, filtering, and grouping, illustrating the power of Python and standardized data access.

**Next Steps**: For in-depth proteomics analysis, consider mapping `@id` fields to semantic concepts (e.g., UniProt accessions, protein types) and integrating domain-specific workflows.